In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import importlib

from functions import color_palettes
importlib.reload(color_palettes)
from functions.color_palettes import *

from functions import utils
importlib.reload(utils)
from functions.utils import *

from functions import config
importlib.reload(config)
from functions.config import *

set_seed(42)
sns.set_theme(style="whitegrid", context="talk")

c:\Users\aoezek\AppData\Local\miniconda3\envs\circadian\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#---------------------------------------------
#Load the cgm data for the complete 46 participants
#---------------------------------------------
cgm_complete, participant_ids, metadata = load_and_filter_complete_cgm(
    cgm_path=cgm_path,
    metadata_path=metadata_path,
    days_to_keep=days,
)
metadata.rename(columns={"ID": "Participant_id"}, inplace=True)
cgm_df = analysis_ready_cgm(cgm_complete)
#----------------------------------------------
#Load the meal data merged with metadata columns age groups
#----------------------------------------------
meal_data = pd.read_excel(merged_meal_meta_valid_path)
cgm_summary_df = pd.read_excel(cgm_extracted_features_path)
meal_events_df = pd.read_excel(models_data_ppgr_path)
age_gender_map = pd.read_excel(age_gender_map_path)
age_gender_map.rename(columns={"ID": "participant_id"}, inplace=True)



In [ ]:
#----------------------------------------------
#Prepare the time series analsis-ready dataset (pre and post points, if necessary eliminations)
#----------------------------------------------
records, exclusion_report = extract_window_dataset(
    meal_events_df = meal_events_df,
    cgm_df = cgm_df,
    cgm_summary_df = cgm_summary_df,
    metadata_df = metadata,
    pre_window_min = 60,
    post_window_min = 120,
    resample_interval_min = 15,
    min_post_points = 6,
    min_pre_points = 2,
    max_gap_min = 25
    )

records_df = pd.DataFrame(records)

# The sequences are stored as lists; for modeling need converting
post_seq = np.stack(records_df["post_glucose_inc"].values)  # post meal 8 points (N, 8)
pre_seq= np.stack(records_df["pre_glucose_inc"].values) # pre meal 5 points(N, 5)

In [ ]:
records_df = engineer_features_task2(records_df)
records_df['participant_id'] = records_df['participant_id'].str.replace('AL', 'P', regex=False)
records_df.to_excel(r"C:\Users\aoezek\Desktop\master_thesis\master_thesis\finalised_clean_notebooks\excels\task2_timeseries_data.xlsx", index=False)

In [17]:
#for visualisation of number of meals left per participant
counts = (records_df.groupby("participant_id").size().reset_index(name="count"))
time_series_data_counts = counts.merge(age_gender_map, on='participant_id', how='left')
time_series_data_counts= time_series_data_counts.sort_values(['count'],ascending=False)

In [ ]:
plt.figure(figsize=(14,8))

ax = sns.barplot(
    data=time_series_data_counts,
    x="participant_id",
    y="count",
    hue='age_group_label',
    palette=[mixed_palette[2], mixed_palette[1]],
    alpha=0.8
)
for p in ax.patches:
    height = p.get_height()
    ax.annotate(
        f"{height:.0f}",
        (p.get_x() + p.get_width()/2, height),
        ha="center",
        va="bottom",
        fontsize=12
    )
plt.xticks(rotation=45, fontsize=12)
plt.ylabel("Number of Meal Images")
plt.xlabel("Participant")
plt.title("Number of Available Meal Images After Merging and Elimination")
plt.legend()
plt.tight_layout()
plt.savefig('C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\number_of_meals_timeseries.png', dpi=400)
plt.show()